In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch
import os


os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# 1. 加载模型和分词器
model_name = "EleutherAI/gpt-neo-125M"
# model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. 加载数据集
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. 分词
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 4. 创建 DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. 创建 DataLoader
tokenized_datasets.set_format("torch")
dataloader = DataLoader(tokenized_datasets, batch_size=10, shuffle=True, collate_fn=data_collator)

# 6. 设置优化器
optimizer = AdamW(model.parameters(), lr=5e-5)

# 7. 训练循环
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# device = torch.device("cpu")
model.to(device)
model.train()

from experiments.trainer.plugins import SnapshotPlugin, ProfilerPlugin
from perf_estimator.config import Config
snap_conf = Config(save2tmp=False)
snapshot = SnapshotPlugin(config=snap_conf)
profiler = ProfilerPlugin(config=snap_conf)

epochs = 1
snapshot.start()
profiler.start()
for epoch in range(epochs):
    for index, batch in enumerate(dataloader):
        snapshot.step()
        profiler.step()
        with torch.set_grad_enabled(True):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            if index == 3:
                break

snapshot.stop()
profiler.stop()

print("训练完成！")

/home/glaswegian/miniconda3/envs/xmem-12.8/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_36229/754432672.py:66: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/aten/src/ATen/native/Scalar.cpp:22.)
  print(f"Epoch {epoch}, Loss: {loss.item()}")
[W407 21:56:10.403863113 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


Epoch 0, Loss: 3.9691247940063477
Epoch 0, Loss: 3.945723056793213
Epoch 0, Loss: 4.011063098907471
Epoch 0, Loss: 4.32389497756958
训练完成！
